# lora-stack-lab Kaggle runner
Set `BRANCH`/`REPO`, enable **Internet** in notebook settings, run **Save & Run All (Commit)**.
Outputs: `results/` JSONs + `ckpt/` (small) + `results_pack.zip` for manual download.

In [ ]:
import os, subprocess, sys

BRANCH  = "main"                # <- change per stage
STAGE   = "stage1"               # <- "reeval" (score saved adapters) or stage prefix of configs/experiments/
REPO    = "P-Bao/lora-stack-lab"  # public repo
COMMIT  = None                    # <- optional: pin exact commit for reproduction
RUN_IDS = ["stage1-L-lr2e-4-r32-s2", "stage1-L-lr2e-4-r32-s3"]  # seeds 2/3 of L* for sigma_L

sh = lambda c: subprocess.run(c, shell=True, check=True)
sh("rm -rf /tmp/repo")  # idempotent: always clone fresh (stale /tmp/repo = stale code)
sh(f"git clone --depth 1 --branch {BRANCH} https://github.com/{REPO}.git /tmp/repo")
if COMMIT:
    sh(f"cd /tmp/repo && git fetch --depth 1 origin {COMMIT} && git checkout {COMMIT}")
os.chdir("/tmp/repo")
sh("bash scripts/bootstrap_kaggle.sh")
sys.path.insert(0, "/tmp/repo/src")

In [ ]:
# STAGE="reeval": re-evaluate saved adapters (no training). Dataset dirs: <run_id>/adapter/
# Kaggle mount styles: /kaggle/input/<slug> (old) or /kaggle/input/datasets/<user>/<slug> (new).
from pathlib import Path
import glob as _glob

ADAPTER_DATASET = "stage1-adapters"   # <- dataset slug; contains <run_id>/adapter/
ADAPTER_ROOTS = [f"/kaggle/input/{ADAPTER_DATASET}",
                 *_glob.glob(f"/kaggle/input/datasets/*/{ADAPTER_DATASET}")]
ADAPTER_ROOT = next((p for p in ADAPTER_ROOTS if Path(p).exists()), None)
print("[nb] adapter root:", ADAPTER_ROOT, "| candidates:", ADAPTER_ROOTS, flush=True)

if STAGE == "reeval":
    assert ADAPTER_ROOT, f"dataset '{ADAPTER_DATASET}' not mounted - add it under Add Input"
    # one shard per T4: reeval.py --shard i/n takes runs with index % n == i
    procs = []
    for gpu in (0, 1):
        env = {**os.environ, "CUDA_VISIBLE_DEVICES": str(gpu)}
        procs.append(subprocess.Popen(
            [sys.executable, "-u", "scripts/reeval.py",
             "--adapters", ADAPTER_ROOT, "--out", "/kaggle/working/results",
             "--batch-size", "16", "--eval-size", "200", "--shard", f"{gpu}/2"],
            env=env))
        print(f"[nb] reeval shard {gpu}/2 -> GPU {gpu}", flush=True)
    codes = [pr.wait() for pr in procs]
    if any(codes):
        print(f"[nb] WARNING: reeval shards failed with codes {codes} - fix and rerun; "
              f"finished runs are skipped on rerun", flush=True)
else:
    ids = " ".join(RUN_IDS)
    sh(f"python scripts/run_queue.py --stage {STAGE} --ids {ids} "
       f"--out /kaggle/working/results --ckpt /kaggle/working/ckpt "
       f"--skip-from /kaggle/input")  # scans previous outputs mounted as input to skip done IDs
    # reeval the fresh step-500 snapshots so cross-seed compare is fair vs s1 (trap 10)
    sh("python scripts/reeval.py --adapters /kaggle/working/ckpt "
       "--out /kaggle/working/results --batch-size 16")


In [ ]:
sh("python scripts/collect_results.py --src /kaggle/working/results --zip /kaggle/working/results_pack.zip")